# 可选实验：带 Regularization 的 Cost 与 Gradient


## 目标

在本实验中，你将：

- 为之前的 linear regression 和 logistic regression 的 cost function 加入 regularization term（正则化项）。
- 加入 regularization 后，重新运行之前的 overfitting 示例。


In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from plt_overfit import overfit_example, output
from lab_utils_common import sigmoid
np.set_printoptions(precision=8)

## 加入 Regularization

<img src="./images/C1_W3_LinearGradientRegularized.png" alt="Linear regression 的 regularized cost 与 gradient" style="width:400px;padding:10px;">
<img src="./images/C1_W3_LogisticGradientRegularized.png" alt="Logistic regression 的 regularized cost 与 gradient" style="width:400px;padding:10px;">

上图展示了 linear regression 和 logistic regression 的 cost 及 gradient。注意：

- 两种模型的 cost function 有明显区别，但加入 regularization 的方式相同。
- 两种模型的 gradient 形式非常相似，区别主要在于 $f_{\mathbf{w},b}$ 的计算方式。


## 带 Regularization 的 Cost Function

### Regularized Linear Regression

其 cost function 为：

$$
J(\mathbf{w},b)=\frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)^2
+\frac{\lambda}{2m}\sum_{j=0}^{n-1}w_j^2 \tag{1}
$$

其中：

$$
f_{\mathbf{w},b}(\mathbf{x}^{(i)})=\mathbf{w}\cdot\mathbf{x}^{(i)}+b \tag{2}
$$

与之前不带 regularization 的公式比较：

$$
J(\mathbf{w},b)=\frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)^2
$$

两者的区别是增加了 $\frac{\lambda}{2m}\sum_{j=0}^{n-1}w_j^2$。这一项鼓励 gradient descent 在拟合数据时使用较小的 weight。通常不对参数 $b$ 进行 regularization，本实验也遵循这一做法。

下面实现公式 (1)、(2)，采用本课程常见的方式：使用 `for` loop 遍历全部 `m` 个样本。


In [ ]:
def compute_cost_linear_reg(X, y, w, b, lambda_ = 1):
    """
    Computes the cost over all examples
    Args:
      X (ndarray (m,n): Data, m examples with n features
      y (ndarray (m,)): target values
      w (ndarray (n,)): model parameters  
      b (scalar)      : model parameter
      lambda_ (scalar): Controls amount of regularization
    Returns:
      total_cost (scalar):  cost 
    """

    m  = X.shape[0]
    n  = len(w)
    cost = 0.
    for i in range(m):
        f_wb_i = np.dot(X[i], w) + b                                   #(n,)(n,)=scalar, see np.dot
        cost = cost + (f_wb_i - y[i])**2                               #scalar             
    cost = cost / (2 * m)                                              #scalar  
 
    reg_cost = 0
    for j in range(n):
        reg_cost += (w[j]**2)                                          #scalar
    reg_cost = (lambda_/(2*m)) * reg_cost                              #scalar
    
    total_cost = cost + reg_cost                                       #scalar
    return total_cost                                                  #scalar

运行下面的代码单元，查看实际结果。


In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5,6)
y_tmp = np.array([0,1,0,1,0])
w_tmp = np.random.rand(X_tmp.shape[1]).reshape(-1,)-0.5
b_tmp = 0.5
lambda_tmp = 0.7
cost_tmp = compute_cost_linear_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)

print("Regularized cost:", cost_tmp)

**预期输出：** Regularized cost 为 `0.07917239320214275`。


### Regularized Logistic Regression

其 cost function 为：

$$
\begin{aligned}
J(\mathbf{w},b)
&=\frac{1}{m}\sum_{i=0}^{m-1}\left[-y^{(i)}\log\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right)
-(1-y^{(i)})\log\left(1-f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right)\right] \\
&\quad+\frac{\lambda}{2m}\sum_{j=0}^{n-1}w_j^2.
\end{aligned}
\tag{3}
$$

其中：

$$
f_{\mathbf{w},b}(\mathbf{x}^{(i)})=\operatorname{sigmoid}(\mathbf{w}\cdot\mathbf{x}^{(i)}+b) \tag{4}
$$

不带 regularization 的 cost function 为：

$$
J(\mathbf{w},b)=\frac{1}{m}\sum_{i=0}^{m-1}\left[-y^{(i)}\log\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right)
-(1-y^{(i)})\log\left(1-f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right)\right]
$$

与 linear regression 一样，区别在于 regularization term $\frac{\lambda}{2m}\sum_{j=0}^{n-1}w_j^2$。它鼓励使用较小的 weight，参数 $b$ 不参与 regularization。


In [ ]:
def compute_cost_logistic_reg(X, y, w, b, lambda_ = 1):
    """
    Computes the cost over all examples
    Args:
    Args:
      X (ndarray (m,n): Data, m examples with n features
      y (ndarray (m,)): target values
      w (ndarray (n,)): model parameters  
      b (scalar)      : model parameter
      lambda_ (scalar): Controls amount of regularization
    Returns:
      total_cost (scalar):  cost 
    """

    m,n  = X.shape
    cost = 0.
    for i in range(m):
        z_i = np.dot(X[i], w) + b                                      #(n,)(n,)=scalar, see np.dot
        f_wb_i = sigmoid(z_i)                                          #scalar
        cost +=  -y[i]*np.log(f_wb_i) - (1-y[i])*np.log(1-f_wb_i)      #scalar
             
    cost = cost/m                                                      #scalar

    reg_cost = 0
    for j in range(n):
        reg_cost += (w[j]**2)                                          #scalar
    reg_cost = (lambda_/(2*m)) * reg_cost                              #scalar
    
    total_cost = cost + reg_cost                                       #scalar
    return total_cost                                                  #scalar

运行下面的代码单元，查看实际结果。


In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5,6)
y_tmp = np.array([0,1,0,1,0])
w_tmp = np.random.rand(X_tmp.shape[1]).reshape(-1,)-0.5
b_tmp = 0.5
lambda_tmp = 0.7
cost_tmp = compute_cost_logistic_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)

print("Regularized cost:", cost_tmp)

**预期输出：** Regularized cost 为 `0.6850849138741673`。


## 带 Regularization 的 Gradient Descent

加入 regularization 后，gradient descent 的基本更新方式不变。重复以下步骤，直到收敛：

$$
\begin{aligned}
w_j &:= w_j-\alpha\frac{\partial J(\mathbf{w},b)}{\partial w_j},\qquad j=0,\ldots,n-1, \\
b &:= b-\alpha\frac{\partial J(\mathbf{w},b)}{\partial b}.
\end{aligned}
\tag{5}
$$

每次迭代必须同时更新所有 $w_j$ 和 $b$。变化的是 gradient 的计算方式。


### 计算带 Regularization 的 Gradient

Linear regression 和 logistic regression 的 gradient 形式几乎相同，区别仅在于 $f_{\mathbf{w},b}$ 的计算：

$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j}
=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)x_j^{(i)}+\frac{\lambda}{m}w_j \tag{6}
$$

$$
\frac{\partial J(\mathbf{w},b)}{\partial b}
=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right) \tag{7}
$$

- $m$ 是训练样本数，$f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是预测值，$y^{(i)}$ 是 target。
- 对于 **linear regression**：$f_{\mathbf{w},b}(\mathbf{x})=\mathbf{w}\cdot\mathbf{x}+b$。
- 对于 **logistic regression**：$z=\mathbf{w}\cdot\mathbf{x}+b$，$f_{\mathbf{w},b}(\mathbf{x})=g(z)$，其中 sigmoid function 为 $g(z)=\frac{1}{1+e^{-z}}$。

Regularization 在 $w_j$ 的 gradient 中增加了 $\frac{\lambda}{m}w_j$。


### Regularized Linear Regression 的 Gradient 函数


In [ ]:
def compute_gradient_linear_reg(X, y, w, b, lambda_): 
    """
    Computes the gradient for linear regression 
    Args:
      X (ndarray (m,n): Data, m examples with n features
      y (ndarray (m,)): target values
      w (ndarray (n,)): model parameters  
      b (scalar)      : model parameter
      lambda_ (scalar): Controls amount of regularization
      
    Returns:
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar):       The gradient of the cost w.r.t. the parameter b. 
    """
    m,n = X.shape           #(number of examples, number of features)
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]                 
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]               
        dj_db = dj_db + err                        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m   
    
    for j in range(n):
        dj_dw[j] = dj_dw[j] + (lambda_/m) * w[j]

    return dj_db, dj_dw

运行下面的代码单元，查看实际结果。


In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5,3)
y_tmp = np.array([0,1,0,1,0])
w_tmp = np.random.rand(X_tmp.shape[1])
b_tmp = 0.5
lambda_tmp = 0.7
dj_db_tmp, dj_dw_tmp =  compute_gradient_linear_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)

print(f"dj_db: {dj_db_tmp}", )
print(f"Regularized dj_dw:\n {dj_dw_tmp.tolist()}", )

**预期输出：**

```text
dj_db: 0.6648774569425726
Regularized dj_dw:
 [0.29653214748822276, 0.4911679625918033, 0.21645877535865857]
```


### Regularized Logistic Regression 的 Gradient 函数


In [ ]:
def compute_gradient_logistic_reg(X, y, w, b, lambda_): 
    """
    Computes the gradient for linear regression 
 
    Args:
      X (ndarray (m,n): Data, m examples with n features
      y (ndarray (m,)): target values
      w (ndarray (n,)): model parameters  
      b (scalar)      : model parameter
      lambda_ (scalar): Controls amount of regularization
    Returns
      dj_dw (ndarray Shape (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar)            : The gradient of the cost w.r.t. the parameter b. 
    """
    m,n = X.shape
    dj_dw = np.zeros((n,))                            #(n,)
    dj_db = 0.0                                       #scalar

    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i],w) + b)          #(n,)(n,)=scalar
        err_i  = f_wb_i  - y[i]                       #scalar
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err_i * X[i,j]      #scalar
        dj_db = dj_db + err_i
    dj_dw = dj_dw/m                                   #(n,)
    dj_db = dj_db/m                                   #scalar

    for j in range(n):
        dj_dw[j] = dj_dw[j] + (lambda_/m) * w[j]

    return dj_db, dj_dw  


运行下面的代码单元，查看实际结果。


In [ ]:
np.random.seed(1)
X_tmp = np.random.rand(5,3)
y_tmp = np.array([0,1,0,1,0])
w_tmp = np.random.rand(X_tmp.shape[1])
b_tmp = 0.5
lambda_tmp = 0.7
dj_db_tmp, dj_dw_tmp =  compute_gradient_logistic_reg(X_tmp, y_tmp, w_tmp, b_tmp, lambda_tmp)

print(f"dj_db: {dj_db_tmp}", )
print(f"Regularized dj_dw:\n {dj_dw_tmp.tolist()}", )

**预期输出：**

```text
dj_db: 0.341798994972791
Regularized dj_dw:
 [0.17380012933994293, 0.32007507881566943, 0.10776313396851499]
```


## 重新运行 Overfitting 示例


In [ ]:
plt.close("all")
display(output)
ofit = overfit_example(True)

在上图中，为之前的示例加入 regularization，尝试以下步骤：

1. 选择 `Categorical`（logistic regression），将 degree 设为 6、lambda 设为 0（不使用 regularization），然后拟合数据。
2. 将 lambda 改为 1，增强 regularization，再次拟合，观察区别。
3. 切换到 `Regression`（linear regression），重复同样的操作。


## 恭喜完成实验！

你已经查看了 linear regression 和 logistic regression 中加入 regularization 的 cost、gradient 函数，并直观理解了 regularization 如何减轻 overfitting。
